## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [7]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  # with tf.GradientTape(persistent=True) as tape:
    # tape.watch(x_left)
    # tape.watch(x_right)
  inputs_left = tf.concat([x_left, t], axis=1)
  inputs_right = tf.concat([x_right, t], axis=1)
  p_left = model(inputs_left)
  p_right = model(inputs_right)
  # p_x_left = tape.gradient(p_left, x_left)
  # p_x_right = tape.gradient(p_right, x_right)
  # del tape
  return tf.reduce_mean(
      tf.square(p_left)
  ) + tf.reduce_mean(
      tf.square(p_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.sin(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    print(f"Epoch {epoch}, Loss: {loss.numpy()}")

Epoch 0, Loss: 1.9346959590911865
Epoch 50, Loss: 0.2606266736984253
Epoch 100, Loss: 0.18926966190338135
Epoch 150, Loss: 0.16195198893547058
Epoch 200, Loss: 0.11715500056743622
Epoch 250, Loss: 0.09808235615491867
Epoch 300, Loss: 0.07803312689065933
Epoch 350, Loss: 0.07465505599975586
Epoch 400, Loss: 0.07329784333705902
Epoch 450, Loss: 0.04878094419836998
Epoch 500, Loss: 0.05042783543467522
Epoch 550, Loss: 0.041675981134176254
Epoch 600, Loss: 0.0367564857006073
Epoch 650, Loss: 0.02926977537572384
Epoch 700, Loss: 0.02794584073126316
Epoch 750, Loss: 0.027676373720169067
Epoch 800, Loss: 0.021948887035250664
Epoch 850, Loss: 0.013983456417918205
Epoch 900, Loss: 0.010402233339846134
Epoch 950, Loss: 0.008785567246377468


In [8]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()